
# Transformer Regression Sanity Check

This notebook benchmarks a **plain supervised regressor** built on the repo's
`ObservationEncoder` against tabular baselines for predicting a **single stellar parameter**
from the observed inputs.

The goal is not to beat the SBI models. It is to answer a narrower question:

- does the transformer encoder learn a useful representation from the missingness-aware photometry/astrometry?
- does that representation outperform strong non-neural tabular baselines on the same task?
- does the answer change when we train on a **naturally imbalanced** subset vs a **uniform age-mass-bin** subset?

The notebook supports:

- multiple targets, e.g. `logAge`, `m_init`, `feh`
- multiple training subsets
- evaluation on both natural-distribution and balanced age-mass-bin test subsets
- optional reuse of an existing `test_indices.npy`

The transformer regressor here reuses the same `ObservationEncoder` class that the SBI models use,
so this is a fairly direct architecture sanity check.


In [ ]:

from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data.py').exists() and (candidate / 'encoder.py').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from the current working directory.')


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from columns import OBS_COLS
from data import DEFAULT_INPUT_COLS, build_sbi_arrays, load_cache_arrays
from encoder import ObservationEncoder
from value_transforms import apply_inverse_value_transforms_numpy

print(f'Repo root: {REPO_ROOT}')
print(f'Torch device available: cuda={torch.cuda.is_available()}')


In [ ]:

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

CACHE_ENV = os.environ.get('SBI_CACHE_PATH')
TEST_INDEX_ENV = os.environ.get('SBI_TEST_INDEX_PATH')

CACHE_PATH = Path(CACHE_ENV).expanduser() if CACHE_ENV else None
TEST_INDEX_PATH = Path(TEST_INDEX_ENV).expanduser() if TEST_INDEX_ENV else None

assert CACHE_PATH is not None and CACHE_PATH.exists(), (
    'Set SBI_CACHE_PATH in the environment or edit this cell to point to build_arrays_cache.npz.'
)
if TEST_INDEX_PATH is not None:
    assert TEST_INDEX_PATH.exists(), f'TEST_INDEX_PATH not found: {TEST_INDEX_PATH}'

TARGET_COLUMNS = ['logAge', 'm_init']
INPUT_COLUMNS = list(DEFAULT_INPUT_COLS)
USE_COLORS = False

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_SPLIT = 0.10 if TEST_INDEX_PATH is None else None
VAL_SPLIT = 0.10

VAL_SUBSET_SIZE = 25_000

TRAIN_SUBSET_SPECS = [
    {
        'name': 'natural_100k',
        'kind': 'random',
        'n_rows': 100_000,
    },
    {
        'name': 'uniform_age_mass_100k',
        'kind': 'uniform_age_mass',
        'n_rows': 100_000,
        'n_age_bins': 25,
        'n_mass_bins': 12,
        'bin_strategy': 'equal_width',
        'replace': True,
    },
]

EVAL_SUBSET_SPECS = [
    {
        'name': 'natural_test',
        'kind': 'random',
        'n_rows': None,
    },
    {
        'name': 'uniform_age_mass_test_50k',
        'kind': 'uniform_age_mass',
        'n_rows': 50_000,
        'n_age_bins': 25,
        'n_mass_bins': 12,
        'bin_strategy': 'equal_width',
        'replace': False,
    },
]

TRANSFORMER_CONFIG = {
    'dim_value': 24,
    'dim_id': 24,
    'dim_error': 16,
    'dim_observed': 8,
    'attn_embed_dim': 128,
    'num_heads': 8,
    'num_layers': 4,
    'widening_factor': 4,
    'dropout': 0.05,
    'use_missingness_context': True,
    'missingness_context_hidden_dim': 64,
    'head_hidden_dim': 256,
    'batch_size': 2048,
    'epochs': 40,
    'lr': 3e-4,
    'lr_min': 1e-5,
    'weight_decay': 1e-4,
    'patience': 8,
    'huber_delta': 1.0,
}

RANDOM_FOREST_CONFIG = {
    'n_estimators': 300,
    'max_depth': None,
    'min_samples_leaf': 2,
    'n_jobs': -1,
    'random_state': SEED,
}

HGB_CONFIG = {
    'loss': 'squared_error',
    'learning_rate': 0.05,
    'max_depth': 8,
    'max_iter': 300,
    'min_samples_leaf': 50,
    'random_state': SEED,
    'early_stopping': False,
}

OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'transformer_regression_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print('CACHE_PATH      =', CACHE_PATH)
print('TEST_INDEX_PATH =', TEST_INDEX_PATH)
print('DEVICE          =', DEVICE)
print('OUTPUT_DIR      =', OUTPUT_DIR)


In [ ]:

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def column_index(columns: list[str], name: str) -> int:
    return {str(c): i for i, c in enumerate(columns)}[str(name)]


def denormalize_columns(cache, values_norm: np.ndarray, columns: list[str]) -> np.ndarray:
    idx = np.asarray([column_index(cache.columns, c) for c in columns], dtype=np.int64)
    arr = np.asarray(values_norm, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    raw = arr * cache.stds[idx] + cache.means[idx]
    raw = apply_inverse_value_transforms_numpy(
        raw,
        transform_names=np.asarray(cache.value_transform_names[idx], dtype=object),
        transform_params=np.asarray(cache.value_transform_params[idx], dtype=np.float32),
    )
    return raw


def denormalize_target(cache, values_norm_1d: np.ndarray, target_col: str) -> np.ndarray:
    return denormalize_columns(cache, np.asarray(values_norm_1d, dtype=np.float32), [target_col]).reshape(-1)


def split_rows(
    n_rows: int,
    *,
    seed: int,
    val_split: float,
    test_split: float | None,
    test_index_path: Path | None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_rows = np.arange(n_rows, dtype=np.int64)

    if test_index_path is not None:
        test_rows = np.unique(np.load(test_index_path).astype(np.int64))
        test_rows = test_rows[(test_rows >= 0) & (test_rows < n_rows)]
        keep_mask = np.ones(n_rows, dtype=bool)
        keep_mask[test_rows] = False
        trainval_rows = all_rows[keep_mask]
    else:
        assert test_split is not None and 0.0 < test_split < 1.0
        trainval_rows, test_rows = train_test_split(
            all_rows,
            test_size=test_split,
            random_state=seed,
        )

    train_rows, val_rows = train_test_split(
        trainval_rows,
        test_size=val_split,
        random_state=seed,
    )
    return (
        np.asarray(train_rows, dtype=np.int64),
        np.asarray(val_rows, dtype=np.int64),
        np.asarray(test_rows, dtype=np.int64),
    )


def _build_edges(values: np.ndarray, n_bins: int, strategy: str) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if strategy == 'quantile':
        edges = np.quantile(values, np.linspace(0.0, 1.0, n_bins + 1))
    elif strategy == 'equal_width':
        vmin = float(values.min())
        vmax = float(values.max())
        if vmax <= vmin:
            vmax = vmin + 1e-6
        edges = np.linspace(vmin, vmax, n_bins + 1)
    else:
        raise ValueError(f'Unknown bin strategy: {strategy}')

    eps = max(np.finfo(np.float64).eps * max(float(np.max(np.abs(values))), 1.0), 1e-12)
    edges = np.asarray(edges, dtype=np.float64)
    for i in range(1, len(edges)):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + eps
    if edges[-1] <= edges[-2]:
        edges[-1] = edges[-2] + eps
    return edges


def prepare_age_mass_bin_state(
    cache,
    row_indices: np.ndarray,
    *,
    n_age_bins: int,
    n_mass_bins: int,
    bin_strategy: str = 'equal_width',
) -> dict:
    rows = np.asarray(row_indices, dtype=np.int64)
    age_mass = denormalize_columns(
        cache,
        cache.values_norm[rows][:, [column_index(cache.columns, 'logAge'), column_index(cache.columns, 'm_init')]],
        ['logAge', 'm_init'],
    )
    age = age_mass[:, 0]
    mass = age_mass[:, 1]

    age_edges = _build_edges(age, n_age_bins, bin_strategy)
    mass_edges = _build_edges(mass, n_mass_bins, bin_strategy)

    age_bin = np.searchsorted(age_edges, age, side='right') - 1
    mass_bin = np.searchsorted(mass_edges, mass, side='right') - 1
    age_bin = np.clip(age_bin, 0, n_age_bins - 1)
    mass_bin = np.clip(mass_bin, 0, n_mass_bins - 1)
    joint = age_bin * n_mass_bins + mass_bin

    bins: dict[int, np.ndarray] = {}
    for b in np.unique(joint):
        bins[int(b)] = rows[joint == b]

    return {
        'rows': rows,
        'joint': joint.astype(np.int64),
        'bins': bins,
        'active_bins': np.asarray(sorted(bins.keys()), dtype=np.int64),
        'age_edges': age_edges,
        'mass_edges': mass_edges,
        'n_age_bins': int(n_age_bins),
        'n_mass_bins': int(n_mass_bins),
    }


def sample_uniform_over_age_mass_bins(
    cache,
    row_indices: np.ndarray,
    *,
    total_size: int,
    n_age_bins: int,
    n_mass_bins: int,
    bin_strategy: str = 'equal_width',
    replace: bool = True,
    seed: int = 0,
) -> np.ndarray:
    if total_size <= 0:
        raise ValueError('total_size must be > 0')

    state = prepare_age_mass_bin_state(
        cache,
        row_indices,
        n_age_bins=n_age_bins,
        n_mass_bins=n_mass_bins,
        bin_strategy=bin_strategy,
    )
    bins = state['bins']
    active_bins = state['active_bins']
    rng = np.random.default_rng(seed)

    if replace:
        chosen_bins = rng.choice(active_bins, size=total_size, replace=True)
        sampled = np.empty(total_size, dtype=np.int64)
        for b in active_bins:
            mask = chosen_bins == b
            n = int(mask.sum())
            if n == 0:
                continue
            sampled[mask] = rng.choice(bins[int(b)], size=n, replace=True)
        return sampled.astype(np.int64)

    per_bin = int(math.ceil(total_size / max(len(active_bins), 1)))
    selected_chunks = []
    leftovers = []
    for b in active_bins:
        rows_b = np.asarray(bins[int(b)], dtype=np.int64)
        take = min(per_bin, len(rows_b))
        if take > 0:
            chosen = rng.choice(rows_b, size=take, replace=False)
            selected_chunks.append(chosen)
            if take < len(rows_b):
                remaining = np.setdiff1d(rows_b, chosen, assume_unique=False)
                if len(remaining) > 0:
                    leftovers.append(remaining)

    if not selected_chunks:
        raise RuntimeError('No active age-mass bins available for sampling.')

    selected = np.concatenate(selected_chunks).astype(np.int64)
    if len(selected) > total_size:
        selected = rng.choice(selected, size=total_size, replace=False).astype(np.int64)
    elif len(selected) < total_size and leftovers:
        remaining_pool = np.concatenate(leftovers).astype(np.int64)
        n_extra = min(total_size - len(selected), len(remaining_pool))
        extra = rng.choice(remaining_pool, size=n_extra, replace=False).astype(np.int64)
        selected = np.concatenate([selected, extra]).astype(np.int64)

    return selected


def select_subset_rows(cache, pool_rows: np.ndarray, spec: dict, *, seed: int) -> np.ndarray:
    kind = spec['kind']
    n_rows = spec.get('n_rows')
    rng = np.random.default_rng(seed)
    pool_rows = np.asarray(pool_rows, dtype=np.int64)

    if kind == 'random':
        if n_rows is None or n_rows >= len(pool_rows):
            return pool_rows.copy()
        return np.sort(rng.choice(pool_rows, size=n_rows, replace=False).astype(np.int64))

    if kind == 'uniform_age_mass':
        assert n_rows is not None, 'uniform_age_mass subsets require n_rows'
        return sample_uniform_over_age_mass_bins(
            cache,
            pool_rows,
            total_size=n_rows,
            n_age_bins=int(spec.get('n_age_bins', 25)),
            n_mass_bins=int(spec.get('n_mass_bins', 12)),
            bin_strategy=str(spec.get('bin_strategy', 'equal_width')),
            replace=bool(spec.get('replace', True)),
            seed=seed,
        )

    raise ValueError(f'Unknown subset kind: {kind}')


def make_matching_val_spec(spec: dict, val_subset_size: int) -> dict:
    val_spec = dict(spec)
    val_spec['name'] = f"{spec['name']}_val"
    val_spec['n_rows'] = val_subset_size
    if val_spec['kind'] == 'uniform_age_mass':
        val_spec['replace'] = False
    return val_spec


def build_subset_map(cache, train_rows, val_rows, test_rows):
    train_subset_rows = {}
    val_subset_rows = {}
    eval_subset_rows = {}

    for i, spec in enumerate(TRAIN_SUBSET_SPECS):
        train_subset_rows[spec['name']] = select_subset_rows(cache, train_rows, spec, seed=SEED + 10 * (i + 1))
        val_spec = make_matching_val_spec(spec, val_subset_size=min(VAL_SUBSET_SIZE, len(val_rows)))
        val_subset_rows[spec['name']] = select_subset_rows(cache, val_rows, val_spec, seed=SEED + 100 + 10 * (i + 1))

    for i, spec in enumerate(EVAL_SUBSET_SPECS):
        eval_subset_rows[spec['name']] = select_subset_rows(cache, test_rows, spec, seed=SEED + 200 + 10 * (i + 1))

    return train_subset_rows, val_subset_rows, eval_subset_rows


def plot_age_mass_distributions(cache, subset_rows: dict[str, np.ndarray], *, title_prefix: str = '') -> None:
    names = list(subset_rows.keys())
    ncols = min(3, max(1, len(names)))
    nrows = int(math.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows), squeeze=False)

    for ax, name in zip(axes.ravel(), names):
        rows = np.asarray(subset_rows[name], dtype=np.int64)
        age_mass = denormalize_columns(
            cache,
            cache.values_norm[rows][:, [column_index(cache.columns, 'logAge'), column_index(cache.columns, 'm_init')]],
            ['logAge', 'm_init'],
        )
        hb = ax.hexbin(age_mass[:, 0], age_mass[:, 1], gridsize=45, mincnt=1, cmap='viridis')
        ax.set_title(f"{title_prefix}{name}\nN={len(rows):,}")
        ax.set_xlabel('logAge')
        ax.set_ylabel('m_init')
        fig.colorbar(hb, ax=ax, label='count')

    for ax in axes.ravel()[len(names):]:
        ax.axis('off')

    fig.tight_layout()
    plt.show()


In [ ]:

# ------------------------------------------------------------------
# Load cache and construct train/val/test pools + subset views
# ------------------------------------------------------------------

set_seed(SEED)
cache = load_cache_arrays(str(CACHE_PATH))
train_pool_rows, val_pool_rows, test_pool_rows = split_rows(
    cache.values_norm.shape[0],
    seed=SEED,
    val_split=VAL_SPLIT,
    test_split=TEST_SPLIT,
    test_index_path=TEST_INDEX_PATH,
)

train_subset_rows, val_subset_rows, eval_subset_rows = build_subset_map(
    cache,
    train_pool_rows,
    val_pool_rows,
    test_pool_rows,
)

print(f'train pool: {len(train_pool_rows):,}')
print(f'val pool:   {len(val_pool_rows):,}')
print(f'test pool:  {len(test_pool_rows):,}')

summary_rows = []
for name, rows in train_subset_rows.items():
    summary_rows.append({'split_group': 'train', 'subset': name, 'n_rows': len(rows)})
for name, rows in val_subset_rows.items():
    summary_rows.append({'split_group': 'val', 'subset': name, 'n_rows': len(rows)})
for name, rows in eval_subset_rows.items():
    summary_rows.append({'split_group': 'eval', 'subset': name, 'n_rows': len(rows)})

display(pd.DataFrame(summary_rows))


In [ ]:

plot_age_mass_distributions(cache, train_subset_rows, title_prefix='train: ')
plot_age_mass_distributions(cache, eval_subset_rows, title_prefix='eval: ')


In [ ]:

# ------------------------------------------------------------------
# Models + training/evaluation helpers
# ------------------------------------------------------------------

class RegressionDataset(Dataset):
    def __init__(self, arrays):
        self.inputs = torch.tensor(arrays.inputs, dtype=torch.float32)
        self.errors = torch.tensor(arrays.input_errors, dtype=torch.float32)
        self.observed = torch.tensor(arrays.input_observed, dtype=torch.float32)
        self.targets = torch.tensor(arrays.theta[:, 0], dtype=torch.float32)

    def __len__(self) -> int:
        return self.inputs.shape[0]

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            'inputs': self.inputs[idx],
            'errors': self.errors[idx],
            'observed': self.observed[idx],
            'target': self.targets[idx],
        }


class TransformerScalarRegressor(nn.Module):
    def __init__(self, input_columns: list[str], *, encoder_cfg: dict, head_hidden_dim: int, dropout: float):
        super().__init__()
        self.encoder = ObservationEncoder(input_columns=input_columns, **encoder_cfg)
        self.head = nn.Sequential(
            nn.LayerNorm(self.encoder.output_dim),
            nn.Linear(self.encoder.output_dim, head_hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, values: torch.Tensor, errors: torch.Tensor, observed: torch.Tensor) -> torch.Tensor:
        h = self.encoder(values, errors, observed)
        return self.head(h).squeeze(-1)


def build_tabular_features(arrays, *, missing_value=np.nan) -> np.ndarray:
    values = np.asarray(arrays.inputs, dtype=np.float32).copy()
    errors = np.asarray(arrays.input_errors, dtype=np.float32).copy()
    observed = np.asarray(arrays.input_observed, dtype=np.float32).copy()

    missing = observed < 0.5
    values[missing] = missing_value
    errors[missing] = missing_value
    return np.concatenate([values, errors, observed], axis=1).astype(np.float32)


def regression_metrics(y_true_norm, y_pred_norm, cache, target_col: str) -> dict[str, float]:
    y_true_norm = np.asarray(y_true_norm, dtype=np.float32).reshape(-1)
    y_pred_norm = np.asarray(y_pred_norm, dtype=np.float32).reshape(-1)
    y_true_phys = denormalize_target(cache, y_true_norm, target_col)
    y_pred_phys = denormalize_target(cache, y_pred_norm, target_col)

    return {
        'rmse_norm': float(np.sqrt(mean_squared_error(y_true_norm, y_pred_norm))),
        'mae_norm': float(mean_absolute_error(y_true_norm, y_pred_norm)),
        'r2_norm': float(r2_score(y_true_norm, y_pred_norm)),
        'rmse_phys': float(np.sqrt(mean_squared_error(y_true_phys, y_pred_phys))),
        'mae_phys': float(mean_absolute_error(y_true_phys, y_pred_phys)),
        'r2_phys': float(r2_score(y_true_phys, y_pred_phys)),
    }


def train_transformer_regressor(train_arrays, val_arrays, input_columns: list[str], *, config: dict, device: str):
    encoder_cfg = {
        'dim_value': config['dim_value'],
        'dim_id': config['dim_id'],
        'dim_error': config['dim_error'],
        'dim_observed': config['dim_observed'],
        'attn_embed_dim': config['attn_embed_dim'],
        'num_heads': config['num_heads'],
        'num_layers': config['num_layers'],
        'widening_factor': config['widening_factor'],
        'dropout': config['dropout'],
        'use_missingness_context': config['use_missingness_context'],
        'missingness_context_hidden_dim': config['missingness_context_hidden_dim'],
    }
    model = TransformerScalarRegressor(
        input_columns=input_columns,
        encoder_cfg=encoder_cfg,
        head_hidden_dim=config['head_hidden_dim'],
        dropout=config['dropout'],
    ).to(device)

    train_ds = RegressionDataset(train_arrays)
    val_ds = RegressionDataset(val_arrays)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config['epochs'],
        eta_min=config['lr_min'],
    )
    loss_fn = nn.HuberLoss(delta=float(config.get('huber_delta', 1.0)))

    best_state = None
    best_val = float('inf')
    no_improve = 0
    history = []

    for epoch in range(config['epochs']):
        model.train()
        train_losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            pred = model(
                batch['inputs'].to(device),
                batch['errors'].to(device),
                batch['observed'].to(device),
            )
            loss = loss_fn(pred, batch['target'].to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu().item()))

        model.eval()
        val_losses = []
        val_true = []
        val_pred = []
        with torch.no_grad():
            for batch in val_loader:
                pred = model(
                    batch['inputs'].to(device),
                    batch['errors'].to(device),
                    batch['observed'].to(device),
                )
                loss = loss_fn(pred, batch['target'].to(device))
                val_losses.append(float(loss.detach().cpu().item()))
                val_true.append(batch['target'].cpu().numpy())
                val_pred.append(pred.cpu().numpy())

        scheduler.step()
        val_true = np.concatenate(val_true)
        val_pred = np.concatenate(val_pred)
        val_mae = float(mean_absolute_error(val_true, val_pred))

        history.append({
            'epoch': epoch + 1,
            'train_loss': float(np.mean(train_losses)),
            'val_loss': float(np.mean(val_losses)),
            'val_mae_norm': val_mae,
            'lr': float(scheduler.get_last_lr()[0]),
        })

        if val_mae < best_val:
            best_val = val_mae
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= config['patience']:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


def predict_transformer(model: nn.Module, arrays, *, batch_size: int, device: str) -> np.ndarray:
    ds = RegressionDataset(arrays)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    preds = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch['inputs'].to(device),
                batch['errors'].to(device),
                batch['observed'].to(device),
            )
            preds.append(pred.cpu().numpy())
    return np.concatenate(preds).astype(np.float32)


def build_baseline_models(seed: int) -> dict[str, object]:
    rf = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('model', RandomForestRegressor(**RANDOM_FOREST_CONFIG)),
    ])
    hgb = HistGradientBoostingRegressor(**HGB_CONFIG)
    return {
        'random_forest': rf,
        'hist_gbdt': hgb,
    }


def run_single_target_experiment(
    cache,
    *,
    target_col: str,
    train_subset_name: str,
    train_rows: np.ndarray,
    val_rows: np.ndarray,
    eval_rows_map: dict[str, np.ndarray],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_arrays = build_sbi_arrays(
        cache,
        row_indices=train_rows,
        input_columns=INPUT_COLUMNS,
        theta_columns=[target_col],
        use_colors=USE_COLORS,
    )
    color_norm_stats = None
    if USE_COLORS and train_arrays.color_means is not None:
        color_norm_stats = (train_arrays.color_means, train_arrays.color_stds)

    val_arrays = build_sbi_arrays(
        cache,
        row_indices=val_rows,
        input_columns=INPUT_COLUMNS,
        theta_columns=[target_col],
        use_colors=USE_COLORS,
        color_norm_stats=color_norm_stats,
    )

    eval_arrays_map = {
        name: build_sbi_arrays(
            cache,
            row_indices=rows,
            input_columns=INPUT_COLUMNS,
            theta_columns=[target_col],
            use_colors=USE_COLORS,
            color_norm_stats=color_norm_stats,
        )
        for name, rows in eval_rows_map.items()
    }

    model_input_columns = list(INPUT_COLUMNS)
    if USE_COLORS and train_arrays.color_names:
        model_input_columns = model_input_columns + list(train_arrays.color_names)

    results = []

    transformer_model, history = train_transformer_regressor(
        train_arrays,
        val_arrays,
        model_input_columns,
        config=TRANSFORMER_CONFIG,
        device=DEVICE,
    )
    for eval_name, eval_arrays in eval_arrays_map.items():
        preds = predict_transformer(
            transformer_model,
            eval_arrays,
            batch_size=TRANSFORMER_CONFIG['batch_size'],
            device=DEVICE,
        )
        metrics = regression_metrics(eval_arrays.theta[:, 0], preds, cache, target_col)
        results.append({
            'target': target_col,
            'train_subset': train_subset_name,
            'eval_subset': eval_name,
            'model': 'transformer_encoder_regressor',
            'n_train': len(train_rows),
            'n_eval': len(eval_arrays.theta),
            **metrics,
        })

    X_train_nan = build_tabular_features(train_arrays, missing_value=np.nan)
    y_train = np.asarray(train_arrays.theta[:, 0], dtype=np.float32)

    baseline_models = build_baseline_models(SEED)
    for model_name, model in baseline_models.items():
        if model_name == 'hist_gbdt':
            model.fit(X_train_nan, y_train)
        else:
            model.fit(X_train_nan, y_train)

        for eval_name, eval_arrays in eval_arrays_map.items():
            X_eval_nan = build_tabular_features(eval_arrays, missing_value=np.nan)
            preds = np.asarray(model.predict(X_eval_nan), dtype=np.float32)
            metrics = regression_metrics(eval_arrays.theta[:, 0], preds, cache, target_col)
            results.append({
                'target': target_col,
                'train_subset': train_subset_name,
                'eval_subset': eval_name,
                'model': model_name,
                'n_train': len(train_rows),
                'n_eval': len(eval_arrays.theta),
                **metrics,
            })

    return pd.DataFrame(results), history


In [ ]:

# ------------------------------------------------------------------
# Run experiments
# ------------------------------------------------------------------

all_results = []
history_frames = []

experiment_jobs = [
    (target_col, subset_name)
    for target_col in TARGET_COLUMNS
    for subset_name in train_subset_rows.keys()
]

for job_idx, (target_col, subset_name) in enumerate(experiment_jobs, start=1):
    print(f"[{job_idx}/{len(experiment_jobs)}] target={target_col} | train_subset={subset_name}")
    result_df, history_df = run_single_target_experiment(
        cache,
        target_col=target_col,
        train_subset_name=subset_name,
        train_rows=train_subset_rows[subset_name],
        val_rows=val_subset_rows[subset_name],
        eval_rows_map=eval_subset_rows,
    )
    history_df = history_df.copy()
    history_df['target'] = target_col
    history_df['train_subset'] = subset_name
    all_results.append(result_df)
    history_frames.append(history_df)

results_df = pd.concat(all_results, ignore_index=True)
history_df = pd.concat(history_frames, ignore_index=True)

results_path = OUTPUT_DIR / 'regression_results.csv'
history_path = OUTPUT_DIR / 'transformer_training_history.csv'
results_df.to_csv(results_path, index=False)
history_df.to_csv(history_path, index=False)

print(f'Saved results to {results_path}')
print(f'Saved history to {history_path}')


In [ ]:

# Main summary table
results_df = results_df.sort_values(['target', 'eval_subset', 'rmse_phys', 'mae_phys'])
display(results_df)

summary = (
    results_df
    .pivot_table(
        index=['target', 'train_subset', 'eval_subset'],
        columns='model',
        values=['rmse_phys', 'mae_phys', 'r2_phys'],
    )
    .sort_index()
)
display(summary)


In [ ]:

# Plot transformer training curves
for (target_col, subset_name), group in history_df.groupby(['target', 'train_subset']):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(group['epoch'], group['train_loss'], label='train')
    axes[0].plot(group['epoch'], group['val_loss'], label='val')
    axes[0].set_title(f'{target_col} | {subset_name} | loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('Huber loss')
    axes[0].legend()

    axes[1].plot(group['epoch'], group['val_mae_norm'])
    axes[1].set_title(f'{target_col} | {subset_name} | val MAE (norm)')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('MAE')
    plt.tight_layout()
    plt.show()



## Notes

A few interpretation points:

- The transformer regressor is intentionally simple: `ObservationEncoder` + small MLP head.
  If this already beats tabular baselines, that is a strong signal that the encoder architecture is learning something useful.
- The baseline models see the **same base information** as the transformer, but flattened into tabular features:
  values, errors, and observed masks.
- `RandomForestRegressor` is run with explicit missingness masks plus constant imputation.
  `HistGradientBoostingRegressor` sees `NaN` values directly and can route on missingness natively.
- The `uniform_age_mass_*` subsets use **equal-width** bins in `(logAge, m_init)` and bin-first sampling.
  That is the regime-emphasizing subset you asked for; it is explicitly not quantile balancing.
- The balanced training subset can contain duplicates when `replace=True`.
  That is intentional: it mirrors a bin-first curriculum / oversampling setup.
- If you want a heavier benchmark, increase `TRAIN_SUBSET_SPECS[*]['n_rows']`, add more targets, or set `USE_COLORS=True`.
- If you want a stricter comparison, add more baselines here, e.g. `ExtraTreesRegressor`, `XGBoost`, or CatBoost.
